# ElasticNet SNP Embedding Baselines

This notebook runs single-gene ElasticNet baselines for cross-individual same-gene expression-difference prediction.

Default scope:
- Genes: `/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding/test.10gene.bed`
- SNP embeddings: `/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding_res_260708/<gene>/<sample>.vcf.pt`
- Feature: flattened per-sample `hidden_states` from each `.vcf.pt`
- Target: sample normalized Monocyte bigWig mean over the transcript 3-prime exon minus the train81 average bigWig mean over the same exon
- Split: fixed individual-level train / validation / test split copied from the previous notebook

Recommended environment: `micromamba run -n cima jupyter lab` or a kernel with `torch`, `numpy`, `pandas`, `pyBigWig`, and `scikit-learn`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch

try:
    import pyBigWig
except ImportError as exc:
    raise ImportError('Install pyBigWig before running this notebook.') from exc

try:
    from sklearn.base import clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import ElasticNet, ElasticNetCV
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
except ImportError as exc:
    raise ImportError('Install scikit-learn before running this notebook.') from exc


In [ ]:
PROJECT_DIR = Path('/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding')
HUMAN_DIR = Path('/mnt/rice/default/Workspace/xuxiaolong/human')
EMBEDDING_ROOT = Path('/mnt/rice/default/Workspace/xuxiaolong/human/SNPembedding_res_260708')

GENE_BED = PROJECT_DIR / 'test.10gene.bed'
GENCODE_GTF = HUMAN_DIR / 'gencode.v49.annotation.sorted.gtf'
SAMPLE_NAME_FILE = PROJECT_DIR / '101samples.name.txt'

BIGWIG_DIR = Path('/mnt/genos100-new/Public/CIMA/norm_CIMA_bw_101')
REFERENCE_AVERAGE_BW = PROJECT_DIR / 'reference_from_train81' / 'train81.average.bw'

OUTPUT_DIR = PROJECT_DIR / 'elasticnet_baselines'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SOURCE = 'vcf_hidden_states'  # Uses per-sample *.vcf.pt['hidden_states']; currently observed shape is n_snps x 1024.
FEATURE_KEY = 'hidden_states'
FEATURE_DTYPE = np.float32

VAL_INDIVIDUALS = ['H005', 'H010', 'H055', 'H102', 'H103', 'H137', 'H198', 'H202', 'H276', 'H319']
TEST_INDIVIDUALS = ['H030', 'H117', 'H118', 'H129', 'H195', 'H197', 'H215', 'H225', 'H261', 'H309']

# ElasticNetCV is fit only on training individuals for each gene.
L1_RATIOS = [0.1, 0.5, 0.9, 1.0]
ALPHAS = np.logspace(-4, 2, 30)
CV_FOLDS = 5
MAX_ITER = 20000
TOL = 1e-4
RANDOM_STATE = 20260711

print(f'Embedding root: {EMBEDDING_ROOT}')
print(f'Gene BED: {GENE_BED}')
print(f'Output dir: {OUTPUT_DIR}')


In [ ]:
def read_sample_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep='\t', header=None, names=['accession', 'sample'])
    df['accession'] = df['accession'].astype(str)
    df['sample'] = df['sample'].astype(str)
    return df

sample_table = read_sample_table(SAMPLE_NAME_FILE)
sample_to_accession = dict(zip(sample_table['sample'], sample_table['accession']))
all_individuals = sample_table['sample'].tolist()

val_set = set(VAL_INDIVIDUALS)
test_set = set(TEST_INDIVIDUALS)
train_individuals = [x for x in all_individuals if x not in val_set and x not in test_set]

assert not (val_set & test_set), 'Validation and test sets overlap.'
assert set(VAL_INDIVIDUALS).issubset(all_individuals), 'Some validation individuals are missing from sample table.'
assert set(TEST_INDIVIDUALS).issubset(all_individuals), 'Some test individuals are missing from sample table.'

print(f'train={len(train_individuals)} val={len(VAL_INDIVIDUALS)} test={len(TEST_INDIVIDUALS)} total={len(all_individuals)}')
print('Note: sample table contains 101 individuals; train split is total minus 10 val minus 10 test.')

def bigwig_path_for_sample(sample: str) -> Path:
    accession = sample_to_accession[sample]
    return BIGWIG_DIR / accession / 're-normalized_Monocyte.total.bw'

missing_bigwig = [sample for sample in all_individuals if not bigwig_path_for_sample(sample).exists()]
assert not missing_bigwig, f'Samples missing normalized bigWig: {missing_bigwig}'
assert REFERENCE_AVERAGE_BW.exists(), REFERENCE_AVERAGE_BW


In [ ]:
@dataclass(frozen=True)
class GeneRecord:
    chrom: str
    start: int
    end: int
    name: str
    embedding_name: str
    strand: str
    tss: int
    transcript_id: str
    expression_regions: tuple[tuple[str, int, int], ...]


def normalize_chrom(chrom: str) -> str:
    chrom = str(chrom)
    return chrom if chrom.startswith('chr') else f'chr{chrom}'


def strip_ensembl_version(value: str) -> str:
    return str(value).split('.', 1)[0]


def transcript_id_from_gene_name(name: str) -> str:
    parts = str(name).split('|')
    for part in parts:
        if part.startswith('ENST'):
            return part
    raise ValueError(f'Could not extract ENST transcript id from gene name: {name}')


def embedding_name_from_bed_name(name: str) -> str:
    parts = str(name).split('|')
    if len(parts) >= 3:
        return f'{parts[0]}_{parts[1]}_{parts[2]}'
    return str(name).replace('|', '_')


def parse_gtf_attributes(attr_text: str) -> dict[str, str]:
    return {key: value for key, value in re.findall(r'(\S+) "([^"]*)"', attr_text)}


def load_transcript_exons(gtf_path: Path, transcript_ids: Iterable[str]) -> dict[str, list[tuple[str, int, int, str]]]:
    requested = {str(tid) for tid in transcript_ids}
    requested_base = {strip_ensembl_version(tid) for tid in requested}
    exons: dict[str, list[tuple[str, int, int, str]]] = {tid: [] for tid in requested}

    with Path(gtf_path).open() as handle:
        for line in handle:
            if not line or line.startswith('#'):
                continue
            fields = line.rstrip('\n').split('\t')
            if len(fields) != 9 or fields[2] != 'exon':
                continue
            attrs = parse_gtf_attributes(fields[8])
            transcript_id = attrs.get('transcript_id')
            if not transcript_id:
                continue
            transcript_base = strip_ensembl_version(transcript_id)
            if transcript_id in requested:
                key = transcript_id
            elif transcript_base in requested_base:
                matches = [tid for tid in requested if strip_ensembl_version(tid) == transcript_base]
                if not matches:
                    continue
                key = matches[0]
            else:
                continue

            chrom = normalize_chrom(fields[0])
            start0 = int(fields[3]) - 1
            end0 = int(fields[4])
            strand = fields[6]
            exons.setdefault(key, []).append((chrom, start0, end0, strand))

    missing = [tid for tid in requested if not exons.get(tid)]
    if missing:
        raise KeyError(f'No exon records found in {gtf_path} for transcript(s): {missing[:10]}')
    return exons


def select_three_prime_exon(exons: list[tuple[str, int, int, str]]) -> tuple[tuple[str, int, int], ...]:
    if not exons:
        raise ValueError('Cannot select 3-prime exon from an empty exon list.')
    strands = {exon[3] for exon in exons}
    if len(strands) != 1:
        raise ValueError(f'Expected one strand per transcript, got {sorted(strands)}')
    strand = next(iter(strands))
    if strand == '+':
        exon = max(exons, key=lambda item: (item[2], item[1]))
    elif strand == '-':
        exon = min(exons, key=lambda item: (item[1], item[2]))
    else:
        raise ValueError(f'Unsupported transcript strand: {strand!r}')
    chrom, start, end, _ = exon
    if end <= start:
        raise ValueError(f'Invalid exon interval for 3-prime exon: {exon}')
    return ((chrom, start, end),)


def read_gene_bed(path: Path, gtf_path: Path) -> list[GeneRecord]:
    raw_records = []
    with path.open() as handle:
        for line in handle:
            if not line.strip() or line.startswith('#'):
                continue
            fields = line.rstrip('\n').split('\t')
            chrom, start, end, name = fields[:4]
            strand = fields[5] if len(fields) > 5 else '+'
            start_i, end_i = int(start), int(end)
            tss = (start_i + end_i) // 2
            transcript_id = transcript_id_from_gene_name(name)
            raw_records.append((normalize_chrom(chrom), start_i, end_i, name, embedding_name_from_bed_name(name), strand, tss, transcript_id))

    transcript_exons = load_transcript_exons(gtf_path, [record[7] for record in raw_records])
    genes: list[GeneRecord] = []
    for chrom, start_i, end_i, name, embedding_name, strand, tss, transcript_id in raw_records:
        exons = transcript_exons[transcript_id]
        expression_regions = select_three_prime_exon(exons)
        exon_strand = exons[0][3]
        genes.append(GeneRecord(chrom, start_i, end_i, name, embedding_name, exon_strand, tss, transcript_id, expression_regions))
    return genes


genes = read_gene_bed(GENE_BED, GENCODE_GTF)
missing_gene_dirs = [g.embedding_name for g in genes if not (EMBEDDING_ROOT / g.embedding_name).is_dir()]
assert not missing_gene_dirs, f'Missing embedding directories: {missing_gene_dirs}'

gene_summary = pd.DataFrame([
    {
        'gene': g.name,
        'embedding_name': g.embedding_name,
        'transcript_id': g.transcript_id,
        'strand': g.strand,
        'expression_regions': ';'.join(f'{chrom}:{start}-{end}' for chrom, start, end in g.expression_regions),
        'selected_bp': sum(end - start for _, start, end in g.expression_regions),
    }
    for g in genes
])
gene_summary


In [ ]:
def bw_mean_regions(bw, regions: Iterable[tuple[str, int, int]]) -> float:
    weighted_sum = 0.0
    total_bp = 0
    for chrom, start, end in regions:
        start_i = max(0, int(start))
        end_i = max(start_i, int(end))
        if end_i <= start_i:
            continue
        values = np.array(bw.values(chrom, start_i, end_i, numpy=True), dtype=np.float32)
        if values.size == 0:
            continue
        values = np.nan_to_num(values, nan=0.0)
        weighted_sum += float(values.sum())
        total_bp += int(values.size)
    if total_bp == 0:
        return float('nan')
    return weighted_sum / total_bp


def compute_targets(samples: list[str], gene: GeneRecord) -> pd.DataFrame:
    rows = []
    with pyBigWig.open(str(REFERENCE_AVERAGE_BW)) as ref_bw:
        reference_value = bw_mean_regions(ref_bw, gene.expression_regions)
    for sample in samples:
        with pyBigWig.open(str(bigwig_path_for_sample(sample))) as sample_bw:
            sample_value = bw_mean_regions(sample_bw, gene.expression_regions)
        rows.append({
            'sample': sample,
            'gene': gene.name,
            'embedding_name': gene.embedding_name,
            'sample_value': sample_value,
            'reference_value': reference_value,
            'target_diff': sample_value - reference_value,
        })
    return pd.DataFrame(rows)


In [ ]:
def sample_pt_path(gene: GeneRecord, sample: str) -> Path:
    path = EMBEDDING_ROOT / gene.embedding_name / f'CIMA-{sample}_CIMA-{sample}.vcf.pt'
    if not path.exists():
        matches = sorted((EMBEDDING_ROOT / gene.embedding_name).glob(f'*{sample}*.vcf.pt'))
        if len(matches) != 1:
            raise FileNotFoundError(f'Expected one .vcf.pt for {gene.embedding_name} / {sample}, found {len(matches)}: {matches[:5]}')
        return matches[0]
    return path


def tensor_to_numpy(value) -> np.ndarray:
    if hasattr(value, 'detach'):
        value = value.detach().cpu().float().numpy()
    else:
        value = np.asarray(value)
    return np.asarray(value, dtype=FEATURE_DTYPE)


def load_sample_feature(gene: GeneRecord, sample: str) -> np.ndarray:
    payload = torch.load(sample_pt_path(gene, sample), map_location='cpu', weights_only=False)
    if not isinstance(payload, dict):
        raise TypeError(f'Expected dict payload in .vcf.pt, got {type(payload)} for {gene.embedding_name} / {sample}')
    if FEATURE_KEY not in payload:
        raise KeyError(f'{FEATURE_KEY!r} is missing from {sample_pt_path(gene, sample)}; keys={sorted(payload.keys())}')
    arr = tensor_to_numpy(payload[FEATURE_KEY])
    if arr.ndim != 2:
        raise ValueError(f'Expected 2D {FEATURE_KEY} for {gene.embedding_name} / {sample}, got shape={arr.shape}')
    if not np.isfinite(arr).all():
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr.reshape(-1)


def load_gene_feature_matrix(gene: GeneRecord, samples: list[str]) -> tuple[np.ndarray, dict[str, int | tuple[int, ...]]]:
    features = []
    shapes = []
    for sample in samples:
        payload = torch.load(sample_pt_path(gene, sample), map_location='cpu', weights_only=False)
        arr = tensor_to_numpy(payload[FEATURE_KEY])
        if arr.ndim != 2:
            raise ValueError(f'Expected 2D {FEATURE_KEY} for {gene.embedding_name} / {sample}, got shape={arr.shape}')
        shapes.append(tuple(arr.shape))
        features.append(np.nan_to_num(arr.reshape(-1), nan=0.0, posinf=0.0, neginf=0.0).astype(FEATURE_DTYPE, copy=False))
    unique_shapes = sorted(set(shapes))
    if len(unique_shapes) != 1:
        raise ValueError(f'Feature shape mismatch for {gene.embedding_name}: {unique_shapes[:10]}')
    X = np.vstack(features)
    info = {
        'feature_shape': unique_shapes[0],
        'n_snps': int(unique_shapes[0][0]),
        'embedding_dim': int(unique_shapes[0][1]),
        'n_features': int(X.shape[1]),
    }
    return X, info

# Lightweight schema check on the first gene / first sample.
example_gene = genes[0]
example_sample = all_individuals[0]
example_payload = torch.load(sample_pt_path(example_gene, example_sample), map_location='cpu', weights_only=False)
print(example_gene.embedding_name, example_sample)
print({k: (tuple(v.shape) if hasattr(v, 'shape') else type(v).__name__) for k, v in example_payload.items()})


In [ ]:
def safe_pearson(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) < 2 or np.std(y_true) == 0 or np.std(y_pred) == 0:
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) == 0 or np.var(y_true) == 0:
        return float('nan')
    return float(r2_score(y_true, y_pred))


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        'rmse': float(math.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'pearson': safe_pearson(y_true, y_pred),
        'r2': safe_r2(y_true, y_pred),
    }


def prefixed_metrics(prefix: str, y_true, y_pred) -> dict[str, float]:
    return {f'{prefix}_{key}': value for key, value in regression_metrics(y_true, y_pred).items()}


def split_arrays(X: np.ndarray, y: np.ndarray, samples: list[str]):
    sample_to_idx = {sample: i for i, sample in enumerate(samples)}
    train_idx = [sample_to_idx[s] for s in train_individuals]
    val_idx = [sample_to_idx[s] for s in VAL_INDIVIDUALS]
    test_idx = [sample_to_idx[s] for s in TEST_INDIVIDUALS]
    return train_idx, val_idx, test_idx


def make_elasticnet_cv() -> Pipeline:
    model = ElasticNetCV(
        l1_ratio=L1_RATIOS,
        alphas=ALPHAS,
        cv=CV_FOLDS,
        max_iter=MAX_ITER,
        tol=TOL,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        selection='cyclic',
    )
    return Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('model', model),
    ])


def refit_fixed_elasticnet(best_pipeline: Pipeline, X_train: np.ndarray, y_train: np.ndarray) -> Pipeline:
    best_model = best_pipeline.named_steps['model']
    fixed = ElasticNet(
        alpha=float(best_model.alpha_),
        l1_ratio=float(best_model.l1_ratio_),
        max_iter=MAX_ITER,
        tol=TOL,
        random_state=RANDOM_STATE,
        selection='cyclic',
    )
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
        ('scaler', StandardScaler(with_mean=True, with_std=True)),
        ('model', fixed),
    ])
    pipe.fit(X_train, y_train)
    return pipe


In [ ]:
def run_single_gene_elasticnet(gene: GeneRecord) -> tuple[dict, pd.DataFrame, dict]:
    samples = all_individuals
    X, feature_info = load_gene_feature_matrix(gene, samples)
    target_df = compute_targets(samples, gene)
    y = target_df['target_diff'].to_numpy(dtype=float)

    train_idx, val_idx, test_idx = split_arrays(X, y, samples)
    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        cv_pipe = make_elasticnet_cv()
        cv_pipe.fit(X_train, y_train)
    convergence_warnings = sum(issubclass(w.category, ConvergenceWarning) for w in caught)

    # Refit on train+val using hyperparameters selected only from train CV. Test remains untouched.
    train_val_idx = train_idx + val_idx
    final_pipe = refit_fixed_elasticnet(cv_pipe, X[train_val_idx], y[train_val_idx])

    pred_train = final_pipe.predict(X_train)
    pred_val = final_pipe.predict(X_val)
    pred_test = final_pipe.predict(X_test)

    final_model = final_pipe.named_steps['model']
    coef = np.asarray(final_model.coef_)
    zero_test = np.zeros_like(y_test, dtype=float)
    train_mean_test = np.full_like(y_test, fill_value=float(np.mean(y_train)), dtype=float)

    row = {
        'gene': gene.name,
        'embedding_name': gene.embedding_name,
        **feature_info,
        'n_train': len(train_idx),
        'n_val': len(val_idx),
        'n_test': len(test_idx),
        'target_train_mean': float(np.mean(y_train)),
        'target_train_std': float(np.std(y_train)),
        'target_val_mean': float(np.mean(y_val)),
        'target_val_std': float(np.std(y_val)),
        'target_test_mean': float(np.mean(y_test)),
        'target_test_std': float(np.std(y_test)),
        'selected_alpha': float(cv_pipe.named_steps['model'].alpha_),
        'selected_l1_ratio': float(cv_pipe.named_steps['model'].l1_ratio_),
        'nonzero_coef': int(np.count_nonzero(coef)),
        'coef_l1': float(np.sum(np.abs(coef))),
        'coef_l2': float(np.sqrt(np.sum(coef ** 2))),
        'intercept': float(final_model.intercept_),
        'convergence_warnings': int(convergence_warnings),
        **prefixed_metrics('train', y_train, pred_train),
        **prefixed_metrics('val', y_val, pred_val),
        **prefixed_metrics('test', y_test, pred_test),
        **prefixed_metrics('zero_baseline_test', y_test, zero_test),
        **prefixed_metrics('train_mean_baseline_test', y_test, train_mean_test),
    }

    prediction_rows = []
    for split_name, idxs, preds in [
        ('train', train_idx, pred_train),
        ('val', val_idx, pred_val),
        ('test', test_idx, pred_test),
    ]:
        for i, pred in zip(idxs, preds):
            prediction_rows.append({
                'split': split_name,
                'sample': samples[i],
                'gene': gene.name,
                'embedding_name': gene.embedding_name,
                'prediction_diff': float(pred),
                'target_diff': float(y[i]),
            })

    coef_summary = {
        'gene': gene.name,
        'embedding_name': gene.embedding_name,
        'n_features': int(coef.size),
        'nonzero_coef': int(np.count_nonzero(coef)),
        'coef_l1': float(np.sum(np.abs(coef))),
        'coef_l2': float(np.sqrt(np.sum(coef ** 2))),
        'coef_abs_max': float(np.max(np.abs(coef))) if coef.size else float('nan'),
    }
    return row, pd.DataFrame(prediction_rows), coef_summary


In [ ]:
metric_rows = []
prediction_frames = []
coef_rows = []

for gene in genes:
    print(f'Running ElasticNet baseline: {gene.embedding_name}')
    row, pred_df, coef_row = run_single_gene_elasticnet(gene)
    metric_rows.append(row)
    prediction_frames.append(pred_df)
    coef_rows.append(coef_row)
    print({
        'gene': gene.embedding_name,
        'n_snps': row['n_snps'],
        'n_features': row['n_features'],
        'alpha': row['selected_alpha'],
        'l1_ratio': row['selected_l1_ratio'],
        'nonzero_coef': row['nonzero_coef'],
        'test_pearson': row['test_pearson'],
        'test_r2': row['test_r2'],
        'test_rmse': row['test_rmse'],
        'zero_baseline_test_rmse': row['zero_baseline_test_rmse'],
    })

metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_frames, ignore_index=True)
coef_summary = pd.DataFrame(coef_rows)

metrics_path = OUTPUT_DIR / 'elasticnet_single_gene_metrics.csv'
predictions_path = OUTPUT_DIR / 'elasticnet_single_gene_predictions.csv'
coef_path = OUTPUT_DIR / 'elasticnet_single_gene_coeff_summary.csv'

metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
coef_summary.to_csv(coef_path, index=False)

print(f'Wrote {metrics_path}')
print(f'Wrote {predictions_path}')
print(f'Wrote {coef_path}')
metrics.sort_values('test_pearson', ascending=False)


In [ ]:
summary_cols = [
    'gene', 'embedding_name', 'n_snps', 'n_features', 'selected_alpha', 'selected_l1_ratio', 'nonzero_coef',
    'target_test_std', 'test_rmse', 'test_mae', 'test_pearson', 'test_r2',
    'zero_baseline_test_rmse', 'zero_baseline_test_mae', 'zero_baseline_test_r2',
    'train_mean_baseline_test_rmse', 'train_mean_baseline_test_mae', 'train_mean_baseline_test_r2',
]
metrics[summary_cols].sort_values('test_pearson', ascending=False)


## Notes for Interpreting the Output

- `test_pearson` and `test_r2` are computed per gene across the fixed test individuals, so they directly measure same-gene cross-individual prediction.
- `zero_baseline_test_*` reports the baseline that predicts expression difference `0` for every test individual.
- `train_mean_baseline_test_*` reports the baseline that predicts the training-individual mean target for that gene.
- If `nonzero_coef` is 0, ElasticNet selected a fully shrunk model for that gene.
- If test performance is not better than the zero or train-mean baseline, the current flattened SNP embedding feature does not provide a useful linear signal for that gene under this split.
